# VietRAG — Colab Demo

Evidence-first RAG for Vietnamese institutional/specialized documents.

**External AI APIs used: Gemini + OpenRouter only.** FAISS/BM25/document parsing run locally.

This notebook never downloads `providers.env` from GitHub because this repository is public. It checks `/content/providers.env`; if the file is missing, Colab asks you to upload your private copy.


In [ ]:
# 1) Clone/update the project
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/NVTruong473/NLP.git"
REPO_DIR = Path("/content/NLP")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# 2) Install dependencies
!pip -q install -r requirements.txt
!pip -q install -e . --no-deps


## 3) Load private API keys safely

Prepare a local `providers.env` from `providers.env.example`. Do **not** commit the real file. If keys have ever appeared in a screenshot or public commit, rotate them first.


In [ ]:
from pathlib import Path
import shutil

ENV_PATH = Path("/content/providers.env")

# Optional: if you already mounted Google Drive and keep a private copy there, use it.
drive_candidate = Path("/content/drive/MyDrive/providers.env")
if not ENV_PATH.exists() and drive_candidate.exists():
    shutil.copy2(drive_candidate, ENV_PATH)

if not ENV_PATH.exists():
    from google.colab import files
    print("providers.env not found. Upload your PRIVATE providers.env now.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No providers.env uploaded.")
    # Accept any selected filename but store it at the expected private runtime path.
    selected_name = next(iter(uploaded))
    Path(selected_name).replace(ENV_PATH)

from vietrag.secrets import load_provider_env, key_summary
load_provider_env(ENV_PATH)
print("Credentials loaded (values hidden):", key_summary())


## 4) Build the knowledge index

The default demo uses official public TDTU admissions/regulation pages. Set `USE_DEMO_SOURCES=False` and upload your own PDF/DOCX/TXT/HTML files to `data/uploads/` for a private-domain demo.


In [ ]:
import subprocess
from pathlib import Path

USE_DEMO_SOURCES = True
REBUILD_INDEX = False
INDEX_FILE = Path("artifacts/index/faiss.index")

if REBUILD_INDEX or not INDEX_FILE.exists():
    cmd = ["python", "scripts/build_index.py", "--env", str(ENV_PATH)]
    if USE_DEMO_SOURCES:
        cmd += ["--sources", "data/demo_sources.yaml"]
    else:
        upload_dir = Path("data/uploads")
        upload_dir.mkdir(parents=True, exist_ok=True)
        from google.colab import files
        print("Upload PDF/DOCX/TXT/MD/HTML documents")
        uploaded_docs = files.upload()
        for name, blob in uploaded_docs.items():
            (upload_dir / Path(name).name).write_bytes(blob)
        cmd += ["--sources", "", "--input-dir", str(upload_dir)]
    subprocess.run(cmd, check=True)
else:
    print("Existing index found; skipping re-embedding to save free-tier quota.")


## 5) Ask questions

The response is generated only after hybrid retrieval passes the domain-confidence gate. Sources and retrieval scores are shown separately.


In [ ]:
from vietrag.config import load_config
from vietrag.pipeline import VietRAGPipeline
from vietrag.providers import GeminiProvider, OpenRouterProvider

cfg = load_config("configs/default.yaml")
rag = VietRAGPipeline.load(cfg, GeminiProvider(), OpenRouterProvider())

question = "TDTU có những phương thức tuyển sinh đại học nào trong năm 2026?"
result = rag.ask(question)
print(result.text)
print(rag.format_sources(result))
print("\nprovider=", result.provider, " top_dense=", round(result.top_dense_score, 4), " refused=", result.refused)


### OOD/refusal check

A specialized knowledge assistant should not answer unrelated questions from model memory.


In [ ]:
ood_question = "Hãy cho tôi công thức làm bánh tiramisu."
ood_result = rag.ask(ood_question)
print(ood_result.text)
print("refused=", ood_result.refused, " top_dense=", round(ood_result.top_dense_score, 4))


## 6) Retrieval/OOD evaluation

The starter file is intentionally small. Expand `evaluation/sample_eval.jsonl` before reporting academic results.


In [ ]:
!python scripts/evaluate.py --env /content/providers.env


## 7) Calibrate the OOD threshold

Do not assume one cosine threshold works for every corpus. The calibration script searches for a threshold that maximizes balanced accuracy on a labeled calibration set. Use a larger held-out set for a real report.


In [ ]:
!python scripts/calibrate_ood.py --env /content/providers.env


## 8) Launch Gradio

The public Gradio share URL exposes the **app**, not your `providers.env`. The keys remain in the temporary Colab runtime. Do not print or display the env file in notebook output.


In [ ]:
!python app.py
